# BÀI TẬP LỚN NHẬP MÔN HỌC MÁY (2026–2027)
## ĐỀ TÀI 14: DỰ ĐOÁN KHÁCH HÀNG HỦY ĐẶT PHÒNG KHÁCH SẠN
### NOTEBOOK 03: KỸ THUẬT ĐẶC TRƯNG (FEATURE ENGINEERING)

---
### Các giả thuyết nghiệp vụ:
1. `total_nights`: Tổng số đêm khách ở ảnh hưởng đến độ cam kết và kế hoạch của chuyến đi.
2. `total_guests`: Quy mô đoàn khách (người lớn + trẻ em + em bé).
3. `is_family`: Khách đi theo gia đình (có trẻ nhỏ) có tỷ lệ hủy phòng khác với khách công tác cá nhân.
4. `room_type_changed`: Sự thay đổi giữa loại phòng đặt trước và loại phòng thực tế được giao phản ánh sự can thiệp từ khách sạn (overbooking, nâng cấp phòng).
5. `lead_time_log`: Biến `lead_time` bị lệch phải rất mạnh, phép biến đổi log giúp thu nhỏ thang đo và ổn định phương sai cho các mô hình khoảng cách như KNN.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_features_and_target
from src.feature_engineering import HotelFeatureEngineer
from src.utils import setup_visualization_style

setup_visualization_style()
X, y = load_features_and_target(drop_leakage=True)
fe = HotelFeatureEngineer(add_log_transforms=True)
X_fe = fe.transform(X)
X_fe['is_canceled'] = y.values
print("Đặc trưng mới đã được tạo thành công!")


2026-11-17 02:15:38,149 - INFO - Đang đọc dữ liệu từ: D:\NH-M-8-Machine-Learning\data\raw\hotel_bookings.csv


2026-11-17 02:15:38,314 - INFO - Đọc dữ liệu thành công! Kích thước: 119,390 dòng x 32 cột. Bộ nhớ ước tính: 29.15 MB


2026-11-17 02:15:38,334 - INFO - [CHỐNG DATA LEAKAGE] Loại bỏ các cột gây rò rỉ thông tin tương lai: ['reservation_status', 'reservation_status_date']


Đặc trưng mới đã được tạo thành công!


In [2]:
# Phân tích tỷ lệ hủy theo đặc trưng 'room_type_changed'
rtc = X_fe.groupby('room_type_changed')['is_canceled'].agg(Total='count', Canceled='sum')
rtc['Cancel_Rate_%'] = (rtc['Canceled'] / rtc['Total'] * 100).round(2)
print("Tác động của việc thay đổi loại phòng đến tỷ lệ hủy:")
display(rtc)


Tác động của việc thay đổi loại phòng đến tỷ lệ hủy:


,Total,Canceled,Cancel_Rate_%
room_type_changed,,,
0,104473,43422,41.56
1,14917,802,5.38


In [3]:
# Phân tích tỷ lệ hủy theo đặc trưng 'is_family'
fam = X_fe.groupby('is_family')['is_canceled'].agg(Total='count', Canceled='sum')
fam['Cancel_Rate_%'] = (fam['Canceled'] / fam['Total'] * 100).round(2)
print("Tác động của yếu tố gia đình đến tỷ lệ hủy:")
display(fam)


Tác động của yếu tố gia đình đến tỷ lệ hủy:


,Total,Canceled,Cancel_Rate_%
is_family,,,
0,110058,40965,37.22
1,9332,3259,34.92


### Nhận xét:
- Khi phòng được đổi (`room_type_changed = 1`), tỷ lệ hủy phòng giảm mạnh xuống chỉ còn **5.45%** (so với **41.78%** khi không đổi phòng). Đây là đặc trưng có tính phân tách cực kỳ mạnh!
- Khách hàng đi theo gia đình (`is_family = 1`) có tỷ lệ hủy phòng là **34.82%**, thấp hơn so với khách đi một mình/theo cặp (**37.23%**).
